### Kernel: Python

## DeepSurv para cancer de mama (covariaveis selecionadas pelo estudo anterior + Optuna)

In [1]:
# Configuracoes e imports
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

SEED = 2003
EPOCHS = 200
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
VALID_FRAC = 0.2
BATCH_SIZE = 128
HIDDEN_DIMS = (64, 32)
DROPOUT = 0.2
PATIENCE = 30

np.random.seed(SEED)
torch.manual_seed(SEED)

## Carregamento dos dados

Usa os arquivos imputados (treino e teste) gerados no pre-processamento.

In [2]:
# Leitura dos dados

df_treino = pd.read_csv("../../../dados/dados-processados/dados_treino.csv")
df_teste = pd.read_csv("../../../dados/dados-processados/dados_teste.csv")

df_treino.head()

,rhc_raca_cor,instrucao,est_cong,rhc_historico_familiar_cancer,rhc_historico_alcool,rhc_historico_tabaco,rhc_origem_encamiamento,rhc_estadiamento_clinico,rhc_primeiro_tratamento_recebido_no_hospital_nenhum,rhc_primeiro_tratamento_recebido_no_hospital_cirurgia,...,macroregiao,pndr_renda,faixa_etcat,ano,tipo_hcido,id_paciente,diff_data_1consulta,diff_data_tratamento,tempo,delta_t
0,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,Sim,...,3,Média Renda,5,2014,8500,1,12,42.0,16,1
1,Não Branca,0,2,Não,Nunca,Sim,SUS,III e IV,Não,Sim,...,3,Média Renda,5,2012,8500,2,-104,69.0,60,1
2,Não Branca,1,2,Não,Nunca,Nunca,SUS,III e IV,Não,Sim,...,1,Baixa Renda,1,2010,8500,5,83,99.0,69,1
3,Não Branca,1,2,Sim,Nunca,Sim,SUS,III e IV,Não,Não,...,1,Alta Renda,2,2017,min100,6,-10,46.0,7,1
4,Não Branca,1,2,Não,Nunca,Nunca,SUS,III e IV,Não,Não,...,3,Média Renda,1,2016,8500,7,52,91.0,27,1


## Preparacao da entrada da rede

Usa as covariaveis do modelo de Cox (vars_paper).
One-hot para categoricas.
Numericas padronizadas com dados da base de treino

In [3]:
# Colunas basicas
COL_ID = "id_paciente"
COL_TIME = "tempo"
COL_EVENT = "delta_t"

# Covariaveis usadas no modelo de Cox (vars_paper)
COX_COVARS = [
    "stage_paper",
    "prior_diag_trat_paper",
    "hormonioterapia_paper",
    "educ_paper",
    "quimioterapia_paper",
    "macroregiao_paper",
]

REQUIRED_RAW_COLS = [
    "rhc_estadiamento_clinico",
    "rhc_diagnostico_e_tratamentos_anteriores",
    "rhc_primeiro_tratamento_recebido_no_hospital_hormonioterapia",
    "rhc_primeiro_tratamento_recebido_no_hospital_quimioterapia",
    "instrucao",
    "macroregiao",
]

def prepare_cox_covariates(df):
    missing = [c for c in REQUIRED_RAW_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Colunas ausentes para recodificacao Cox: {missing}")

    df = df.copy()

    stage_raw = df["rhc_estadiamento_clinico"]
    df["stage_paper"] = stage_raw.map({"I e II": "I/II", "III e IV": "III/IV"})

    prior_raw = df["rhc_diagnostico_e_tratamentos_anteriores"]
    df["prior_diag_trat_paper"] = pd.Series(pd.NA, index=df.index, dtype="string")
    df.loc[prior_raw == "Com diag./Com trat.", "prior_diag_trat_paper"] = "Com diag./Com trat."
    df.loc[prior_raw.isin(["Com diag./Sem trat.", "Sem diag./Sem trat."]), "prior_diag_trat_paper"] = "Outros"

    horm_raw = df["rhc_primeiro_tratamento_recebido_no_hospital_hormonioterapia"]
    df["hormonioterapia_paper"] = horm_raw.map({"Sim": "Sim", "Não": "Nao", "Nao": "Nao"})

    quimio_raw = df["rhc_primeiro_tratamento_recebido_no_hospital_quimioterapia"]
    df["quimioterapia_paper"] = quimio_raw.map({"Sim": "Sim", "Não": "Nao", "Nao": "Nao"})

    educ_raw = df["instrucao"]
    df["educ_paper"] = pd.Series(pd.NA, index=df.index, dtype="string")
    df.loc[educ_raw == 0, "educ_paper"] = "None"
    df.loc[educ_raw.isin([1, 2]), "educ_paper"] = "Primary/Secondary+"

    macro_raw = df["macroregiao"]
    df["macroregiao_paper"] = pd.Series(pd.NA, index=df.index, dtype="string")
    df.loc[macro_raw == 2, "macroregiao_paper"] = "2"
    df.loc[macro_raw.isin([1, 3]), "macroregiao_paper"] = "1/3"

    df = df[(df[COL_TIME] > 0) & (df[COL_EVENT].notna())]
    df = df.dropna(subset=COX_COVARS)
    return df

df_treino_modelo = prepare_cox_covariates(df_treino)
df_teste_modelo = prepare_cox_covariates(df_teste)

feature_cols = COX_COVARS

# Split treino/validacao (estratificado pelo evento)
df_tr, df_val = train_test_split(
    df_treino_modelo,
    test_size=VALID_FRAC,
    random_state=SEED,
    stratify=df_treino_modelo[COL_EVENT],
)

def build_design_matrices(df_train, df_other, feature_cols):
    X_train_raw = df_train[feature_cols].copy()
    X_other_raw = df_other[feature_cols].copy()

    # Detecta colunas numericas no treino
    numeric_cols = X_train_raw.select_dtypes(include=[np.number]).columns.tolist()

    # One-hot nas categoricas (dummy_na para consistencia)
    X_train = pd.get_dummies(X_train_raw, dummy_na=True)
    X_other = pd.get_dummies(X_other_raw, dummy_na=True)

    # Alinha colunas com base no treino (categorias novas no teste sao ignoradas)
    X_train, X_other = X_train.align(X_other, join="left", axis=1, fill_value=0.0)

    # Padroniza apenas colunas numericas originais (presentes em X_train)
    stats = {}
    for col in numeric_cols:
        if col in X_train.columns:
            mean = X_train[col].mean()
            std = X_train[col].std(ddof=0)
            std = std if std > 0 else 1.0
            X_train[col] = (X_train[col] - mean) / std
            X_other[col] = (X_other[col] - mean) / std
            stats[col] = {"mean": float(mean), "std": float(std)}

    return X_train, X_other, stats

X_tr, X_val, stats_num = build_design_matrices(df_tr, df_val, feature_cols)
_, X_test, _ = build_design_matrices(df_tr, df_teste_modelo, feature_cols)

# Tensores para treinamento
t_tr = df_tr[COL_TIME].astype(float).values
e_tr = df_tr[COL_EVENT].astype(int).values
t_val = df_val[COL_TIME].astype(float).values
e_val = df_val[COL_EVENT].astype(int).values

X_tr_t = torch.tensor(X_tr.values.astype(np.float32))
X_val_t = torch.tensor(X_val.values.astype(np.float32))

print(f"X_tr shape: {X_tr_t.shape}, eventos: {e_tr.sum()}")

X_tr shape: torch.Size([3888, 18]), eventos: 460


## Definicao do modelo

A saida e o log-risco (log hazard ratio). A funcao de perda e a log-verossimilhanca parcial de Cox (forma negativa).

$$Perda = - \sum_{i: E_i=1} \left( \hat{h}_i - \log \sum_{j \in R(t_i)} e^{\hat{h}_j} \right)$$

In [5]:
class DeepSurvNet(nn.Module):
    def __init__(self, input_dim, hidden_dims=(64, 32), dropout=0.2):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)

def neg_partial_log_likelihood(log_h, times, events):
    # Ordena por tempo decrescente para construir conjuntos de risco
    order = np.argsort(-times)
    log_h = log_h[order]
    events = events[order]

    risk = torch.exp(log_h)
    log_cum_risk = torch.log(torch.cumsum(risk, dim=0))
    event_mask = torch.tensor(events.astype(np.float32), device=log_h.device)

    # Soma apenas nos eventos observados
    loss = -torch.sum((log_h - log_cum_risk) * event_mask)
    denom = event_mask.sum() + 1e-8
    return loss / denom

def concordance_index(times, events, risks):
    # Concordancia simples (sem empates complexos)
    n = 0
    n_conc = 0
    for i in range(len(times)):
        for j in range(i + 1, len(times)):
            if events[i] == 1 and times[i] < times[j]:
                n += 1
                n_conc += 1 if risks[i] > risks[j] else 0
            elif events[j] == 1 and times[j] < times[i]:
                n += 1
                n_conc += 1 if risks[j] > risks[i] else 0
    return n_conc / n if n > 0 else np.nan

## Otimização dos hiperparâmetros

In [6]:
# Optuna (opcional)
import optuna

RUN_OPTUNA = True
N_TRIALS = 100
MAX_EPOCHS_OPTUNA = 300
PATIENCE_OPTUNA = 15

hidden_dims_map = {
    "32-16": (32, 16),
    "64-32": (64, 32),
    "128-64": (128, 64),
    "128-64-32": (128, 64, 32)
}

dropout_grid = [0.0, 0.1, 0.2, 0.3, 0.4]

def train_val_loss(model, optimizer, X_train, t_train, e_train, X_valid, t_valid, e_valid, batch_size):
    dataset = TensorDataset(X_train, torch.tensor(t_train), torch.tensor(e_train))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=False)
    model.train()
    for xb, tb, eb in loader:
        optimizer.zero_grad()
        log_h = model(xb)
        loss = neg_partial_log_likelihood(log_h, tb.numpy(), eb.numpy())
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        log_h_val = model(X_valid)
        val_loss = neg_partial_log_likelihood(log_h_val, t_valid, e_valid).item()
    return val_loss

def objective(trial):
    hidden_dims_key = trial.suggest_categorical("hidden_dims", list(hidden_dims_map.keys()))
    hidden_dims = hidden_dims_map[hidden_dims_key]
    dropout = trial.suggest_categorical("dropout", dropout_grid)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128, 256])
    lr = trial.suggest_float("learning_rate", 1e-4, 5e-3, log=True)
    wd = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    max_epochs = trial.suggest_int("epochs", 80, MAX_EPOCHS_OPTUNA)
    model = DeepSurvNet(input_dim=X_tr_t.shape[1], hidden_dims=hidden_dims, dropout=dropout)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    best_loss = np.inf
    best_epoch = 0
    for epoch in range(max_epochs):
        val_loss = train_val_loss(model, optimizer, X_tr_t, t_tr, e_tr, X_val_t, t_val, e_val, batch_size)
        if val_loss < best_loss - 1e-5:
            best_loss = val_loss
            best_epoch = epoch
        if (epoch - best_epoch) >= PATIENCE_OPTUNA:
            break
    return best_loss

if RUN_OPTUNA:
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=N_TRIALS)
    best = study.best_params
    HIDDEN_DIMS = hidden_dims_map[best["hidden_dims"]]
    DROPOUT = float(best["dropout"])
    BATCH_SIZE = int(best["batch_size"])
    LEARNING_RATE = float(best["learning_rate"])
    WEIGHT_DECAY = float(best["weight_decay"])
    EPOCHS = int(best["epochs"])
    print(study.best_params)

[I 2026-05-30 12:14:55,735] A new study created in memory with name: no-name-aca50ae8-81aa-47e2-a39d-b3a66d9805ec
[I 2026-05-30 12:15:04,455] Trial 0 finished with value: 6.284585475921631 and parameters: {'hidden_dims': '128-64', 'dropout': 0.0, 'batch_size': 64, 'learning_rate': 0.0009023157681458292, 'weight_decay': 1.3452955824729504e-05, 'epochs': 275}. Best is trial 0 with value: 6.284585475921631.
[I 2026-05-30 12:15:56,190] Trial 1 finished with value: 6.2903733253479 and parameters: {'hidden_dims': '128-64-32', 'dropout': 0.3, 'batch_size': 32, 'learning_rate': 0.00012507584861628298, 'weight_decay': 8.180293659099299e-06, 'epochs': 300}. Best is trial 0 with value: 6.284585475921631.
[I 2026-05-30 12:16:04,542] Trial 2 finished with value: 6.286220073699951 and parameters: {'hidden_dims': '64-32', 'dropout': 0.2, 'batch_size': 128, 'learning_rate': 0.00022357693364003536, 'weight_decay': 1.0227007507640645e-05, 'epochs': 130}. Best is trial 0 with value: 6.284585475921631.
[I

{'hidden_dims': '64-32', 'dropout': 0.0, 'batch_size': 32, 'learning_rate': 0.000910735722364508, 'weight_decay': 3.1557248133726227e-06, 'epochs': 252}


In [7]:


































# Inicializacao e treino (com early stopping)
model = DeepSurvNet(input_dim=X_tr_t.shape[1], hidden_dims=HIDDEN_DIMS, dropout=DROPOUT)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

times_tr = df_tr[COL_TIME].astype(float).values
events_tr = df_tr[COL_EVENT].astype(int).values
times_val = df_val[COL_TIME].astype(float).values
events_val = df_val[COL_EVENT].astype(int).values

train_ds = TensorDataset(X_tr_t, torch.tensor(times_tr), torch.tensor(events_tr))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)

best_state = None
best_val = np.inf
patience = 0

for epoch in range(EPOCHS):
    model.train()
    for xb, tb, eb in train_loader:
        optimizer.zero_grad()
        log_h = model(xb)
        loss = neg_partial_log_likelihood(log_h, tb.numpy(), eb.numpy())
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_loss = neg_partial_log_likelihood(model(X_val_t), times_val, events_val).item()
        risk_tr = torch.exp(model(X_tr_t)).cpu().numpy()
        risk_val = torch.exp(model(X_val_t)).cpu().numpy()
    c_tr = concordance_index(times_tr, events_tr, risk_tr)
    c_val = concordance_index(times_val, events_val, risk_val)
    if (epoch + 1) % 25 == 0:
        print(f"Epoch {epoch + 1:3d}/{EPOCHS} | val_loss={val_loss:.4f} | c_tr={c_tr:.4f} | c_val={c_val:.4f}")
    if val_loss < best_val - 1e-5:
        best_val = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience = 0
    else:
        patience += 1
        if patience >= PATIENCE:
            print("Early stopping acionado.")
            break

if best_state is not None:
    model.load_state_dict(best_state)

Epoch  25/252 | val_loss=6.2938 | c_tr=0.6914 | c_val=0.6929
Early stopping acionado.


## Baseline survival (Breslow) e predicao

Usa o conjunto de treino para estimar a funcao de sobrevivencia baseline. S0(t)

$$H_0(t) = H_0(t-1) + \frac{d_t}{\sum_{j \in R(t)} \exp(\hat{h}_j)}$$

$$S_0(t) = e^{-H_0(t)}$$

In [8]:
def breslow_baseline(times, events, risks):
    # risks = exp(log_hazard) no treino
    order = np.argsort(times)
    times = times[order]
    events = events[order]
    risks = risks[order]

    unique_times = np.unique(times[events == 1])
    cum_hazard = []
    h = 0.0
    for t in unique_times:
        idx_event = (times == t) & (events == 1)
        d_t = idx_event.sum()
        if d_t == 0:
            continue
        risk_set = risks[times >= t].sum()
        h += d_t / max(risk_set, 1e-12)
        cum_hazard.append(h)
    return unique_times, np.array(cum_hazard)

def baseline_survival_at(query_times, base_times, cum_hazard):
    query_times = np.asarray(query_times, dtype=float)
    surv = np.ones_like(query_times, dtype=float)
    if len(base_times) == 0:
        return surv
    mask = query_times >= base_times[0]
    idx = np.searchsorted(base_times, query_times[mask], side="right") - 1
    idx = np.clip(idx, 0, len(cum_hazard) - 1)
    surv[mask] = np.exp(-cum_hazard[idx])
    return surv

# Baseline a partir do treino
model.eval()
with torch.no_grad():
    risk_tr = torch.exp(model(X_tr_t)).cpu().numpy()

base_times, cum_hazard = breslow_baseline(
    times_tr.astype(float),
    events_tr.astype(int),
    risk_tr.astype(float)
)

## Predicao no grid TIME=1..131 e exportacao CSV

Gera a saida no formato ID, TIME, PRED_S conforme os demais modelos.

$$S(t|x) = S_0(t)^{\exp(\hat{h}(x))}$$

In [9]:
# Grid de tempos para avaliacao
tempos_avaliacao_teste = np.arange(1, 132, 1)

# Base por paciente no teste
base_ids = df_teste_modelo[[COL_ID] + feature_cols].drop_duplicates().sort_values(COL_ID)
X_ids = pd.get_dummies(base_ids[feature_cols], dummy_na=True)

# Alinha com as colunas do treino (mesmas covariaveis/one-hot)
X_ids, _ = X_ids.align(pd.DataFrame(columns=X_tr.columns), join="right", axis=1, fill_value=0.0)

# Padroniza numericas com stats do treino
for col, stats in stats_num.items():
    if col in X_ids.columns:
        X_ids[col] = (X_ids[col] - stats["mean"]) / stats["std"]

X_ids_t = torch.tensor(X_ids.values.astype(np.float32))
with torch.no_grad():
    risk_ids = torch.exp(model(X_ids_t)).cpu().numpy().reshape(-1)

# Baseline survival no grid
S0_grid = baseline_survival_at(tempos_avaliacao_teste, base_times, cum_hazard)

# Monta saida ID, TIME, PRED_S
linhas = []
for idx, id_pac in enumerate(base_ids[COL_ID].values):
    s_ind = S0_grid ** risk_ids[idx]
    for t, s in zip(tempos_avaliacao_teste, s_ind):
        linhas.append({
            "ID": int(id_pac),
            "TIME": float(t),
            "PRED_S": float(s)
        })

pred_out = pd.DataFrame(linhas).sort_values(["ID", "TIME"]).reset_index(drop=True)

dir_saida = Path("../dados/saida")
dir_saida.mkdir(parents=True, exist_ok=True)
arquivo_saida = dir_saida / "predicao-deepsurv-optuna-covariaveis-selecionadas.csv"
pred_out.to_csv(arquivo_saida, index=False)

arquivo_saida

WindowsPath('../dados/saida/predicao-deepsurv-optuna-covariaveis-selecionadas.csv')